# 11 · Second backbone: bge-reranker-base

**Question:** does a cross-encoder from a different model family match DeBERTa-v3-base, and does it add to it? Training DeBERTa for a second epoch did not help (val −0.0003, test −0.0008), but averaging the 1-epoch and 2-epoch models did (+0.0012 test, CI excludes 0) although their scores agree closely (Spearman 0.97). A model that ranks more differently should add more.

**Model:** `BAAI/bge-reranker-base`, an XLM-RoBERTa-base cross-encoder already trained for query–passage relevance (one-logit head). It differs from DeBERTa in pre-training, tokenizer and attention, and is cheaper per step (plain attention, the extra parameters are mostly the 250k-token embedding table).

| Run | What | Splits |
|---|---|---|
| `d1_bgerr_main_zeroshot` | pretrained one-logit head, no training (smoke only) | val |
| `d1_bgerr_main_ce4_s42` | fine-tuned, everything else as `d2_deberta_main_ce4_s42` | val, test (blind) |

**Settings** (= `d2_deberta_main_ce4_s42`, only the model changes): `main` text, `max_len` 256, ce4 loss (new 4-class output layer; the pretrained dense layer of the head is kept), 1 epoch, batch 32, lr 2e-5, 1,200 warm-up steps, fp16, seed 42.

**Field separator.** Product text joins fields with `[SEP]`, a special token for DeBERTa. XLM-R has no `[SEP]` and would read it as plain characters, so it is replaced by the model's own separator `</s>` (checked below to be a single token).

**Compared with:** `d2_deberta_main_ce4_s42` on val (paired bootstrap), plus the plain mean of both models' scores (both are expected gains on the same 0–1 scale). Test stays blind; `10_report` scores it.

**Kaggle settings:** GPU **T4**, **Internet on**.

**Inputs:**
- Dataset `wm-src`, version with `predict(..., return_probs=True)` (19/11) and `data/results/` holding `d2_deberta_main_ce4_s42`. Required.
- Notebook Output → **01_preprocess** (`pairs.parquet`, `products_clean.parquet`). Required.

**How to run.**
1. `MODE = 'smoke'`: zero-shot on val, then 5% of train (about 20–25 minutes). Check the verdict at the end of section 2: loss falls, val above the DeBERTa smoke (0.8705) or close to it, and the estimated full-epoch time.
2. `MODE = 'full'`, then **Save Version → Save & Run All** (about 1.5–2.5 h, the smoke prints the estimate). Download `results/` into `data/results/` in the repo.

**Output** (`/kaggle/working/results/`): `<run>.json` and `scores/<run>_{val,test}.parquet` (with `p_E..p_I` for the fine-tuned run).

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import gc, inspect, os, time

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; before torch

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.cross_encoder import predict, train_ce
from src.data import load_pairs
from src.evaluation import RunLog, paired_bootstrap, summarize
from src.text import SEP

assert 'return_probs' in inspect.signature(predict).parameters, 'wm-src is an old version: update the input'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
transformers.logging.set_verbosity_error()

MODE = 'smoke'   # 'smoke': zero-shot + 5% of train, val only | 'full': all train, val + test (blind)

MODEL = 'BAAI/bge-reranker-base'
CFG = {'epochs': 1, 'batch_size': 32, 'weight_decay': 0.01, 'max_grad_norm': 1.0, 'fp16': True,
       'model': MODEL, 'fields': 'main', 'max_len': 256, 'lr': 2e-5, 'warmup_steps': 1200,
       'eval_batch_size': 128, 'loss': 'ce4'}            # = d2_deberta_main_ce4_s42 except the model
NAME = 'd1_bgerr_main'
REF = 'd2_deberta_main_ce4_s42'
SMOKE_REF = 0.8705                                      # val nDCG of the DeBERTa main ce4 smoke (5% of train)

SEED = 42
SUBSAMPLE = 0.05 if MODE == 'smoke' else 1.0
EVAL_SPLITS = ['val'] if MODE == 'smoke' else ['val', 'test']   # test is predicted and saved, never scored here
EVAL_EVERY = 300 if MODE == 'smoke' else 4000                    # steps between val checks (monitoring only)
RES = Path('/kaggle/working/results')

assert MODE in ('smoke', 'full')
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4'
print('torch', torch.__version__, '| transformers', transformers.__version__, '|', torch.cuda.get_device_name(0))
print('MODE', MODE, '| model', MODEL)

## 1. Data

Same pairs as every run, `main` product text, with the field separator switched to the tokenizer's own. In smoke mode, 5% of the training queries.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
sep = f' {tok.sep_token} '
ids = tok('red shoe', 'nike' + sep + 'color: red')['input_ids']
assert ids.count(tok.sep_token_id) == 4, tok.convert_ids_to_tokens(ids)   # </s></s> between the pair, 1 inside, 1 at the end
print('separator', repr(SEP), '->', repr(sep), '|', tok.convert_ids_to_tokens(ids))

pairs = load_pairs(CFG['fields'])
pairs['text'] = pairs['text'].str.replace(SEP, sep, regex=False)
print('texts still containing [SEP]:', int(pairs['text'].str.contains('[SEP]', regex=False).sum()))

MASK = {s: (pairs['split'] == s).to_numpy() for s in ['val', 'test']}
train_df = pairs[pairs['split'] == 'train']
if SUBSAMPLE < 1:
    keep = train_df['query_id'].drop_duplicates().sample(frac=SUBSAMPLE, random_state=SEED)
    train_df = train_df[train_df['query_id'].isin(keep)]
train_df = train_df.reset_index(drop=True)
parts = {s: pairs[MASK[s]] for s in EVAL_SPLITS}

n_tok = [len(x) for x in tok(list(parts['val']['query'].iloc[:2000]), list(parts['val']['text'].iloc[:2000]))['input_ids']]
print(f'train {train_df["query_id"].nunique():,} queries / {len(train_df):,} pairs | '
      f'val {MASK["val"].sum():,} | test {MASK["test"].sum():,} pairs | '
      f'tokens (2,000 val pairs): median {int(np.median(n_tok))}, > {CFG["max_len"]}: {np.mean(np.array(n_tok) > CFG["max_len"]):.1%}')

## 2. Runs

Smoke mode first scores the pretrained model as it is (one logit, no training) on val: a check that the text format suits it, and the zero-shot reference for the report. Then the fine-tuned run, logged with `RunLog`; `d2_deberta_main_ce4_s42` is loaded from `wm-src` for the paired comparison on val.

In [ ]:
log = RunLog(RES, {s: parts[s][['example_id', 'query_id', 'esci_label']] for s in EVAL_SPLITS})
have_ref = log.load(REF, splits=['val'])
if not have_ref:
    print(f'{REF}: not found in the inputs (no paired comparison)')

if MODE == 'smoke':
    t0 = time.time()
    zs = AutoModelForSequenceClassification.from_pretrained(MODEL).float().to('cuda')
    sc = predict(zs, tok, parts['val'], CFG['max_len'], batch_size=CFG['eval_batch_size'], fp16=CFG['fp16'])
    log.log(f'smoke_{NAME}_zeroshot', {'val': sc}, {'mode': MODE, 'model': MODEL, 'fields': CFG['fields'],
            'max_len': CFG['max_len'], 'predict_minutes': round((time.time() - t0) / 60, 1)},
            compare_to=REF if have_ref else None)
    del zs
    gc.collect()
    torch.cuda.empty_cache()

name = f"{'smoke_' if MODE == 'smoke' else ''}{NAME}_ce4_s{SEED}"
print(f'\n==== {name}')
model, tokenizer, info = train_ce(CFG, train_df, SEED, eval_df=parts['val'], eval_every=EVAL_EVERY,
                                  print_every=100 if MODE == 'smoke' else 1000)
t0 = time.time()
torch.cuda.empty_cache()
scores, probs = {}, {}
for s in EVAL_SPLITS:
    scores[s], probs[s] = predict(model, tokenizer, parts[s], CFG['max_len'], batch_size=CFG['eval_batch_size'],
                                  fp16=CFG['fp16'], return_probs=True)
log.log(name, scores, {'mode': MODE, **CFG, 'field_sep': tok.sep_token, 'seed': SEED, 'subsample': SUBSAMPLE,
                       'train_queries': int(train_df['query_id'].nunique()), 'train_pairs': len(train_df),
                       'gpu': torch.cuda.get_device_name(0), 'predict_minutes': round((time.time() - t0) / 60, 1),
                       **info},
        compare_to=REF if have_ref else None, blind=('test',), extra=probs)
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
if MODE == 'smoke':
    r = log.records[name]
    first, last = r['loss_log'][0]['loss'], r['loss_log'][-1]['loss']
    v = r['metrics']['val']['ndcg']
    zs_v = log.records[f'smoke_{NAME}_zeroshot']['metrics']['val']['ndcg']
    full_h = r['steps'] / SUBSAMPLE / r['it_per_s'] / 3600
    ok = last < first and v > 0.742
    print(f"zero-shot val nDCG {zs_v} | smoke val nDCG {v} (DeBERTa smoke {SMOKE_REF}) | "
          f"loss {first:.4f} -> {last:.4f} | full epoch ~{full_h:.2f} h + ~{r['predict_minutes'] * 5.4:.0f} min predict")
    print('Smoke test passed: switch MODE to full.' if ok else 'Run did not learn: send me this output.')

## 3. Results (val)

`paired_vs`: val nDCG difference with `d2_deberta_main_ce4_s42` over the same queries. The mean of both models' scores shows whether the second backbone adds to DeBERTa; if it does clearly (CI excludes 0), the next step is 5-fold OOF scores for it as a feature group of the LightGBM re-ranker (notebooks 07–09).

In [ ]:
display(log.table(('ndcg', 'ndcg_ci95', 'ndcg10', 'mrr_e', 'e_over_s')))
log.print_paired()

curve = pd.DataFrame(log.records[name]['val_curve'])
if len(curve):
    display(curve.set_index('step'))

hits = glob.glob(f'/kaggle/input/**/scores/{REF}_val.parquet', recursive=True)
if hits:
    val = parts['val'][['example_id', 'query_id', 'esci_label']].reset_index(drop=True)
    ref = val[['example_id']].merge(pd.read_parquet(hits[0])[['example_id', 'score']], on='example_id', how='left')['score'].to_numpy()
    new = scores['val']
    print(f'Spearman with {REF} on val: {pd.Series(new).corr(pd.Series(ref), method="spearman"):.3f}')
    m_mean, pq_mean = summarize(val, (new + ref) / 2)
    print(f'mean of both, val nDCG {m_mean["ndcg"]} {m_mean["ndcg_ci95"]} | E-over-S {m_mean["e_over_s"]}')
    print(f'mean - {REF}:', paired_bootstrap(log.per_query[REF]['val'], pq_mean))
    print(f'mean - {name}:', paired_bootstrap(log.per_query[name]['val'], pq_mean))